In [ ]:
# Data Importing

import pandas as pd

file_path = '/content/flights.csv'
df = pd.read_csv(file_path)
display(df.head())

,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,...,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour,name
0,0,2013,1,1,517.0,515,2.0,830.0,819,11.0,...,1545,N14228,EWR,IAH,227.0,1400,5,15,2013-01-01 05:00:00,United Air Lines Inc.
1,1,2013,1,1,533.0,529,4.0,850.0,830,20.0,...,1714,N24211,LGA,IAH,227.0,1416,5,29,2013-01-01 05:00:00,United Air Lines Inc.
2,2,2013,1,1,542.0,540,2.0,923.0,850,33.0,...,1141,N619AA,JFK,MIA,160.0,1089,5,40,2013-01-01 05:00:00,American Airlines Inc.
3,3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,...,725,N804JB,JFK,BQN,183.0,1576,5,45,2013-01-01 05:00:00,JetBlue Airways
4,4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,...,461,N668DN,LGA,ATL,116.0,762,6,0,2013-01-01 06:00:00,Delta Air Lines Inc.


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336776 entries, 0 to 336775
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              336776 non-null  int64  
 1   year            336776 non-null  int64  
 2   month           336776 non-null  int64  
 3   day             336776 non-null  int64  
 4   dep_time        328521 non-null  float64
 5   sched_dep_time  336776 non-null  int64  
 6   dep_delay       328521 non-null  float64
 7   arr_time        328063 non-null  float64
 8   sched_arr_time  336776 non-null  int64  
 9   arr_delay       327346 non-null  float64
 10  carrier         336776 non-null  object 
 11  flight          336776 non-null  int64  
 12  tailnum         334264 non-null  object 
 13  origin          336776 non-null  object 
 14  dest            336776 non-null  object 
 15  air_time        327346 non-null  float64
 16  distance        336776 non-null  int64  
 17  hour      

In [ ]:
df.isnull().sum()

,0
id,0
year,0
month,0
day,0
dep_time,8255
sched_dep_time,0
dep_delay,8255
arr_time,8713
sched_arr_time,0
arr_delay,9430


In [ ]:
# Data Cleaning

df = df.dropna()

In [ ]:
df.isnull().sum()

,0
id,0
year,0
month,0
day,0
dep_time,0
sched_dep_time,0
dep_delay,0
arr_time,0
sched_arr_time,0
arr_delay,0


In [ ]:
display(df.shape)

(327346, 21)

In [ ]:
# Feature Engineering

# 1.Date Column
df['date'] = pd.to_datetime(df[['day','month', 'year']]).dt.strftime('%d-%m-%Y')

# 2.Day of week Column
df['Day_of_Week'] = pd.to_datetime(df['date'], format='%d-%m-%Y').dt.dayofweek

# 3.Weekend is or not column
df['Is_weekend'] = df['Day_of_Week'].apply(lambda x: 1 if x >= 5 else 0)

# 4.Departure period column
def get_departure_period(dep_time):
    if 0 <= dep_time < 500: # 00:00 to 04:59
        return 'Night'
    elif 500 <= dep_time < 1100: # 05:00 to 10:59
        return 'Morning'
    elif 1100 <= dep_time < 1700: # 11:00 to 16:59
        return 'Afternoon'
    elif 1700 <= dep_time < 2100: # 17:00 to 20:59
        return 'Evening'
    else: # 21:00 to 23:59
        return 'Late Night'
df['departure_period'] = df['dep_time'].apply(get_departure_period)

# 5.Departure Delay Flag Column
df['Departure_Delay_Flag'] = df['dep_delay'].apply(lambda x: 0 if x <=0 else 1)

# 6.Arrival Delay Flag Column
df['Arrival_Delay_Flag'] = df['arr_delay'].apply(lambda x: 0 if x <=0 else 1)

# 7.Delay Severity Column
def get_delay_severity(arr_delay):
  if arr_delay <=0:
    return 'On Time'
  elif arr_delay <=30:
    return 'Slight Delay'
  elif arr_delay <=60:
    return 'Medium Delay'
  else:
    return 'Severe Delay'

df['Delay_Severity']=df['arr_delay'].apply(get_delay_severity)

# 8. Route Column
df['Route'] = df['origin'] + '-' + df['dest']

# 9. Short or long distance column
def get_distance_category(distance):
  if distance <=500:
    return 'Short Distance'
  elif 500 < distance <=1000:
    return 'Medium Distance'
  else:
    return 'Long Distance'
df['Distance_Category'] = df['distance'].apply(get_distance_category)

# 10. Hour Column
df['hour'] = pd.to_datetime(df['sched_dep_time'], format='%H%M').dt.hour

# 11. Delay Cause Column
def assign_delay_cause(arr_delay):
    if arr_delay <= 0:
        return "On-time"
    elif arr_delay <= 15:
        return "Minor Operational Delay"
    elif arr_delay <= 45:
        return "Ground Operations Delay"
    elif arr_delay <= 90:
        return "Air Traffic Control Delay"
    elif arr_delay <= 180:
        return "Aircraft / Maintenance Issue"
    else:
        return "Weather / Major Disruption"


df['delay_cause'] = df['arr_delay'].apply(assign_delay_cause)

display(df.head())

,id,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,...,date,Day_of_Week,Is_weekend,departure_period,Departure_Delay_Flag,Arrival_Delay_Flag,Delay_Severity,Route,Distance_Category,delay_cause
0,0,2013,1,1,517.0,515,2.0,830.0,819,11.0,...,01-01-2013,1,0,Morning,1,1,Slight Delay,EWR-IAH,Long Distance,Minor Operational Delay
1,1,2013,1,1,533.0,529,4.0,850.0,830,20.0,...,01-01-2013,1,0,Morning,1,1,Slight Delay,LGA-IAH,Long Distance,Ground Operations Delay
2,2,2013,1,1,542.0,540,2.0,923.0,850,33.0,...,01-01-2013,1,0,Morning,1,1,Medium Delay,JFK-MIA,Long Distance,Ground Operations Delay
3,3,2013,1,1,544.0,545,-1.0,1004.0,1022,-18.0,...,01-01-2013,1,0,Morning,0,0,On Time,JFK-BQN,Long Distance,On-time
4,4,2013,1,1,554.0,600,-6.0,812.0,837,-25.0,...,01-01-2013,1,0,Morning,0,0,On Time,LGA-ATL,Medium Distance,On-time


In [ ]:

output_file_path = 'flights_cleaned.csv'

df.to_csv(output_file_path, index=False)